In [2]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
import joblib

In [4]:
# 1. Define schema & load training data
index_cols = ['unit_nr', 'time_cycles']
setting_cols = ['setting_1', 'setting_2', 'setting_3']
sensor_cols = [f's_{i}' for i in range(1, 22)]
col_names = index_cols + setting_cols + sensor_cols

In [5]:
df = pd.read_csv('data/train_FD001.txt', sep=r'\s+', header=None, names=col_names)

In [16]:
# 2. Calculate target RUL per engine
max_cycles = df.groupby('unit_nr')['time_cycles'].transform('max')
df['RUL'] = (max_cycles - df['time_cycles']).clip(upper=125)

In [11]:
pd.set_option('display.max_columns', None)

In [13]:
df.head(20)

,unit_nr,time_cycles,setting_1,setting_2,setting_3,s_1,s_2,s_3,s_4,s_5,s_6,s_7,s_8,s_9,s_10,s_11,s_12,s_13,s_14,s_15,s_16,s_17,s_18,s_19,s_20,s_21,RUL
0,1,1,-0.0007,-0.0004,100.0,518.67,641.82,1589.70,1400.60,14.62,21.61,554.36,2388.06,9046.19,1.3,47.47,521.66,2388.02,8138.62,8.4195,0.03,392,2388,100.0,39.06,23.4190,125
1,1,2,0.0019,-0.0003,100.0,518.67,642.15,1591.82,1403.14,14.62,21.61,553.75,2388.04,9044.07,1.3,47.49,522.28,2388.07,8131.49,8.4318,0.03,392,2388,100.0,39.00,23.4236,125
2,1,3,-0.0043,0.0003,100.0,518.67,642.35,1587.99,1404.20,14.62,21.61,554.26,2388.08,9052.94,1.3,47.27,522.42,2388.03,8133.23,8.4178,0.03,390,2388,100.0,38.95,23.3442,125
3,1,4,0.0007,0.0000,100.0,518.67,642.35,1582.79,1401.87,14.62,21.61,554.45,2388.11,9049.48,1.3,47.13,522.86,2388.08,8133.83,8.3682,0.03,392,2388,100.0,38.88,23.3739,125
4,1,5,-0.0019,-0.0002,100.0,518.67,642.37,1582.85,1406.22,14.62,21.61,554.00,2388.06,9055.15,1.3,47.28,522.19,2388.04,8133.80,8.4294,0.03,393,2388,100.0,38.90,23.4044,125
5,1,6,-0.0043,-0.0001,100.0,518.67,642.10,1584.47,1398.37,14.62,21.61,554.67,2388.02,9049.68,1.3,47.16,521.68,2388.03,8132.85,8.4108,0.03,391,2388,100.0,38.98,23.3669,125
6,1,7,0.0010,0.0001,100.0,518.67,642.48,1592.32,1397.77,14.62,21.61,554.34,2388.02,9059.13,1.3,47.36,522.32,2388.03,8132.32,8.3974,0.03,392,2388,100.0,39.10,23.3774,125
7,1,8,-0.0034,0.0003,100.0,518.67,642.56,1582.96,1400.97,14.62,21.61,553.85,2388.00,9040.80,1.3,47.24,522.47,2388.03,8131.07,8.4076,0.03,391,2388,100.0,38.97,23.3106,125
8,1,9,0.0008,0.0001,100.0,518.67,642.12,1590.98,1394.80,14.62,21.61,553.69,2388.05,9046.46,1.3,47.29,521.79,2388.05,8125.69,8.3728,0.03,392,2388,100.0,39.05,23.4066,125
9,1,10,-0.0033,0.0001,100.0,518.67,641.71,1591.24,1400.46,14.62,21.61,553.59,2388.05,9051.70,1.3,47.03,521.79,2388.06,8129.38,8.4286,0.03,393,2388,100.0,38.95,23.4694,125


In [14]:
df.tail(20)

,unit_nr,time_cycles,setting_1,setting_2,setting_3,s_1,s_2,s_3,s_4,s_5,s_6,s_7,s_8,s_9,s_10,s_11,s_12,s_13,s_14,s_15,s_16,s_17,s_18,s_19,s_20,s_21,RUL
20611,100,181,0.0024,-0.0005,100.0,518.67,643.25,1597.83,1414.63,14.62,21.61,552.24,2388.18,9065.34,1.3,47.99,520.42,2388.22,8139.30,8.5518,0.03,396,2388,100.0,38.52,23.1774,19
20612,100,182,0.0007,-0.0001,100.0,518.67,643.52,1604.31,1417.73,14.62,21.61,551.47,2388.19,9063.62,1.3,47.75,520.05,2388.22,8140.87,8.4855,0.03,396,2388,100.0,38.41,23.1289,18
20613,100,183,-0.0011,-0.0002,100.0,518.67,643.34,1594.60,1427.27,14.62,21.61,551.83,2388.20,9066.08,1.3,48.00,520.64,2388.23,8144.21,8.5006,0.03,395,2388,100.0,38.49,23.0709,17
20614,100,184,0.0027,-0.0004,100.0,518.67,642.91,1598.88,1420.89,14.62,21.61,551.72,2388.25,9064.14,1.3,47.98,520.05,2388.20,8142.28,8.4989,0.03,396,2388,100.0,38.44,23.1229,16
20615,100,185,-0.0014,0.0004,100.0,518.67,643.95,1600.81,1420.34,14.62,21.61,551.92,2388.19,9069.95,1.3,47.75,520.71,2388.20,8142.32,8.4804,0.03,395,2388,100.0,38.60,23.2127,15
20616,100,186,0.0026,0.0004,100.0,518.67,643.61,1593.55,1425.32,14.62,21.61,552.28,2388.20,9066.46,1.3,48.06,520.66,2388.27,8138.08,8.4735,0.03,394,2388,100.0,38.51,23.1173,14
20617,100,187,0.0015,0.0002,100.0,518.67,643.63,1596.96,1421.49,14.62,21.61,551.53,2388.22,9065.94,1.3,48.04,520.15,2388.22,8140.49,8.5087,0.03,396,2388,100.0,38.67,23.2308,13
20618,100,188,-0.0008,-0.0002,100.0,518.67,643.19,1597.77,1426.57,14.62,21.61,550.77,2388.26,9067.68,1.3,48.29,519.98,2388.21,8139.94,8.4814,0.03,395,2388,100.0,38.36,23.0552,12
20619,100,189,0.0015,0.0001,100.0,518.67,643.69,1599.85,1423.15,14.62,21.61,551.61,2388.18,9069.69,1.3,47.99,519.50,2388.24,8139.78,8.4870,0.03,397,2388,100.0,38.65,23.0591,11
20620,100,190,-0.0001,0.0002,100.0,518.67,643.12,1594.45,1426.04,14.62,21.61,551.06,2388.21,9064.74,1.3,47.99,519.52,2388.26,8142.28,8.5162,0.03,395,2388,100.0,38.42,23.0603,10


In [15]:
same_values = df.columns[df.iloc[0] == df.iloc[-1]]
print(df[same_values].iloc[[0, -1]])

       setting_3     s_1    s_5    s_6  s_10  s_16  s_18   s_19
0          100.0  518.67  14.62  21.61   1.3  0.03  2388  100.0
20630      100.0  518.67  14.62  21.61   1.3  0.03  2388  100.0


The sensor data columns above are uninformative because they are constant and don't change. Therefore, we will drop them.

In [17]:
# 3. Drop uninformative constant sensors
drop_cols = ['s_1', 's_5', 's_6', 's_10', 's_16', 's_18', 's_19']
features = [col for col in setting_cols + sensor_cols if col not in drop_cols]

In [18]:
# 4. Train XGBoost model
X, y = df[features], df['RUL']
model = XGBRegressor(n_estimators=100, learning_rate=0.05, max_depth=5, random_state=42)
model.fit(X, y)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.05, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=5,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=100,
             n_jobs=None, num_parallel_tree=None, ...)

In [19]:
# 5. Save model and list of features
joblib.dump(model, 'model.pkl')
joblib.dump(features, 'features.pkl')
print("Model trained and saved as model.pkl!")

Model trained and saved as model.pkl!
